# M3 — Modelo Avanzado (NCF) vs Baseline ALS — Entrega 3
## Sistema de Recomendación Paralelo para E-Commerce — RetailRocket Dataset

**Este notebook corre en Google Colab con GPU activada**
(`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`).



## 0. Configuración inicial

### Paso A  Verificar GPU


In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU activa. Ve a Entorno de ejecución -> Cambiar tipo de entorno -> GPU, y reinicia.")


### Paso B  Traer el código del repositorio

Clona el repositorio de GitHub para tener los módulos de M1/M2/M3 ya
probados, sin tener que subirlos archivo por archivo.


In [ ]:
!git clone https://github.com/DavidMoraV/Proyecto-Paralela_E-commerce.git
%cd Proyecto-Paralela_E-commerce/Codigos
!pip install -q polars implicit scikit-learn threadpoolctl onnx onnxscript


### Paso C  Subir los datos procesados

Sube `eventos_limpios.parquet` (comprimido en un .zip, ya que es una carpeta
particionada). En tu máquina local, antes de subir:

```powershell
Compress-Archive -Path "datos\eventos_limpios.parquet" -DestinationPath "eventos_limpios.zip"
```


In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

Path("datos").mkdir(exist_ok=True)

print("Sube eventos_limpios.zip:")
subido = files.upload()
with zipfile.ZipFile(list(subido.keys())[0]) as z:
    z.extractall("datos")

print("\nArchivos en datos/:")
!ls -la datos/


## 1. Cargar datos y reconstruir el split train/test (idéntico al de Entrega 2)

In [ ]:
import sys
sys.path.append("src")

from pipeline_datos import cargar_eventos_procesados
from modelo_recomendacion import construir_matriz_interacciones, dividir_train_test, ModeloALS
from pathlib import Path

eventos = cargar_eventos_procesados(Path("datos")).collect()
train, test = dividir_train_test(eventos)
mi = construir_matriz_interacciones(train)

print(f"Usuarios: {mi.matriz.shape[0]:,} | Productos: {mi.matriz.shape[1]:,}")
print(f"Train: {train.height:,} filas | Test: {test.height:,} filas")


## 2. Preparar datos para NCF (pares positivos + mapeo de índices)

Reutiliza los mismos índices usuario/producto que ya construyó `construir_matriz_interacciones`,
para que ALS y NCF trabajen sobre exactamente el mismo espacio de usuarios/productos.


In [ ]:
import numpy as np
from modelo_avanzado import NeuMF, InteraccionesImplicitasDataset, entrenar_ncf, evaluar_ncf_muestreado, exportar_onnx

# Pares positivos de train, ya traducidos a indices contiguos
train_pd = train.select(["visitorid", "itemid"]).to_pandas()
train_pd["u_idx"] = train_pd["visitorid"].map(mi.visitorid_a_idx)
train_pd["i_idx"] = train_pd["itemid"].map(mi.itemid_a_idx)
train_pd = train_pd.dropna()

pares_positivos = train_pd[["u_idx", "i_idx"]].astype(int).to_numpy()

items_por_usuario = {}
for u, i in pares_positivos:
    items_por_usuario.setdefault(int(u), set()).add(int(i))

n_usuarios, n_items = mi.matriz.shape
print(f"Pares positivos de entrenamiento: {len(pares_positivos):,}")


## 3. Entrenar NCF (GPU)

In [ ]:
dataset_ncf = InteraccionesImplicitasDataset(
    pares_positivos, n_usuarios, n_items, items_por_usuario, n_negativos=4
)
print(f"Dataset de entrenamiento (positivos+negativos): {len(dataset_ncf):,} muestras")

modelo_ncf = NeuMF(n_usuarios, n_items, dim_gmf=32, dim_mlp=32, capas_mlp=(64, 32, 16))

resultado_entrenamiento = entrenar_ncf(
    modelo_ncf, dataset_ncf, epochs=10, batch_size=4096, lr=0.001
)
print(f"\nTiempo de entrenamiento (GPU): {resultado_entrenamiento['tiempo_segundos']:.2f} s")
print(f"Dispositivo usado: {resultado_entrenamiento['device']}")


## 4. Benchmark GPU vs CPU

Repite el entrenamiento (menos épocas, solo para medir tiempo por época) en
CPU, para cuantificar la aceleración real de GPU en este modelo.


In [ ]:
modelo_ncf_cpu = NeuMF(n_usuarios, n_items, dim_gmf=32, dim_mlp=32, capas_mlp=(64, 32, 16))
resultado_cpu = entrenar_ncf(modelo_ncf_cpu, dataset_ncf, epochs=2, batch_size=4096, lr=0.001, device="cpu")

tiempo_por_epoca_gpu = resultado_entrenamiento["tiempo_segundos"] / 10
tiempo_por_epoca_cpu = resultado_cpu["tiempo_segundos"] / 2

import pandas as pd
df_gpu_vs_cpu = pd.DataFrame([
    {"dispositivo": "GPU", "tiempo_por_epoca_s": tiempo_por_epoca_gpu},
    {"dispositivo": "CPU", "tiempo_por_epoca_s": tiempo_por_epoca_cpu},
])
df_gpu_vs_cpu["speedup_gpu"] = (df_gpu_vs_cpu["tiempo_por_epoca_s"].iloc[1] / df_gpu_vs_cpu["tiempo_por_epoca_s"]).round(2)
df_gpu_vs_cpu.to_csv("resultados/entrega3/benchmark_gpu_vs_cpu_ncf.csv", index=False)
df_gpu_vs_cpu


## 5. Evaluación comparativa  mismo protocolo para ALS y NCF

Se evalúan ambos modelos con el protocolo muestreado (1 positivo + 99
negativos), para que la comparación sea metodológicamente justa. Esto es
distinto del protocolo de catálogo completo usado para ALS en la Entrega 2
(se documenta la diferencia en el informe).


In [ ]:
# Entrenar baseline ALS aqui tambien, para evaluarlo bajo el MISMO protocolo
modelo_als = ModeloALS(factors=64, regularization=0.01, iterations=15)
modelo_als.entrenar(mi.matriz)

# Adaptar la evaluacion muestreada tambien para ALS
def evaluar_als_muestreado(modelo_als, matriz_train, test_pares, items_por_usuario, mi, n_items, k=10, n_negativos=99):
    rng = np.random.default_rng(42)
    hits, ap_scores, ndcg_scores = [], [], []
    for u, item_real in test_pares:
        vistos = items_por_usuario.get(u, set())
        negativos = []
        while len(negativos) < n_negativos:
            cand = int(rng.integers(0, n_items))
            if cand not in vistos and cand != item_real:
                negativos.append(cand)
        candidatos = np.array(negativos + [int(item_real)])

        # IMPORTANTE: con items=candidatos, Implicit devuelve `ids` YA
        # reordenados por score descendente -- no es un paralelo 1:1 con el
        # array `candidatos` original. Buscar la posicion directamente en `ids`.
        ids, _ = modelo_als.modelo.recommend(
            u, matriz_train[u], N=len(candidatos), items=candidatos, filter_already_liked_items=False
        )
        posicion_real = int(np.where(ids == item_real)[0][0]) + 1

        if posicion_real <= k:
            hits.append(1); ap_scores.append(1.0/posicion_real); ndcg_scores.append(1.0/np.log2(posicion_real+1))
        else:
            hits.append(0); ap_scores.append(0.0); ndcg_scores.append(0.0)
    n = len(hits)
    return {"usuarios_evaluados": n, f"hit_rate@{k}": float(np.mean(hits)),
            f"map@{k}": float(np.mean(ap_scores)), f"ndcg@{k}": float(np.mean(ndcg_scores))}

test_pd = test.select(["visitorid", "itemid"]).to_pandas()
test_pd["u_idx"] = test_pd["visitorid"].map(mi.visitorid_a_idx)
test_pd["i_idx"] = test_pd["itemid"].map(mi.itemid_a_idx)
test_pd = test_pd.dropna()
test_pares = test_pd[["u_idx", "i_idx"]].astype(int).to_numpy()[:5000]  # muestra para evaluacion rapida

metricas_als = evaluar_als_muestreado(modelo_als, mi.matriz, test_pares, items_por_usuario, mi, n_items, k=10, n_negativos=99)
metricas_ncf = evaluar_ncf_muestreado(modelo_ncf, test_pares, items_por_usuario, n_items, k=10, n_negativos=99)

print("ALS (protocolo muestreado):", metricas_als)
print("NCF (protocolo muestreado):", metricas_ncf)


In [ ]:
import json

comparacion = {
    "als_muestreado": metricas_als,
    "ncf_muestreado": metricas_ncf,
    "entrenamiento_ncf": {
        "tiempo_segundos": resultado_entrenamiento["tiempo_segundos"],
        "device": resultado_entrenamiento["device"],
        "historial_perdida": resultado_entrenamiento["historial_perdida"],
    },
    "benchmark_gpu_vs_cpu": df_gpu_vs_cpu.to_dict(orient="records"),
}

with open("resultados/entrega3/comparacion_als_vs_ncf.json", "w") as f:
    json.dump(comparacion, f, indent=2, ensure_ascii=False)

print(json.dumps(comparacion, indent=2, ensure_ascii=False))


## 6. Exportar el modelo final a ONNX

In [ ]:
Path("modelos").mkdir(exist_ok=True)
exportar_onnx(modelo_ncf, "modelos/modelo_avanzado.onnx", device="cuda" if torch.cuda.is_available() else "cpu")

import os
print(f"Tamano del modelo exportado: {os.path.getsize('modelos/modelo_avanzado.onnx') / 1024:.1f} KB")


## 7. Descargar resultados

Descarga estos 3 archivos y colócalos en tu carpeta local
(`Codigos/resultados/entrega3/` y `Codigos/modelos/`) para seguir trabajando
en VS Code.


In [ ]:
from google.colab import files

files.download("resultados/entrega3/comparacion_als_vs_ncf.json")
files.download("resultados/entrega3/benchmark_gpu_vs_cpu_ncf.csv")
files.download("modelos/modelo_avanzado.onnx")
